# 日内波动与震荡强度报表

回答："这段时间里，哪些证券日内波动足够大、来回震荡多、扣除成本后有可交易的空间"。
**打分只用常规时段（9:30–16:00）**；跳空、盘前盘后只作参考。
规范：`docs/intraday_volatility_report_spec.md`（v3）；计算逻辑在 `trader.analytics`，这里只设参数、看结果。

**用法**：改下面参数单元格里的证券和日期，然后菜单 *Run → Run All Cells*。
只能分析本地已有数据的证券；缺数据的会在摘要里列出，请先用数据管理（`run_downloader.bat`）下载。

In [ ]:
# ===== 参数 =====
from trader.analytics import ReportParams

tickers = [
    "NVDA", "AMD", "TSLA", "AAPL", "META", "AMZN", "GOOG", "MSFT", "SMH", "IWM",
    "DIA", "GLD", "TLT", "XLE", "XLF", "KORU", "NVDL", "TSLL", "METU", "SNDK",
]  # fmt: skip
start_date = "2026-04-01"
end_date = "2026-09-30"

params = ReportParams(
    benchmark_tickers=("SPY", "QQQ"),  # 参照证券，始终加入计算
    adr_days=21,  # ADR：过去多少个交易日常规时段振幅的平均
    swing_thresholds_adr=(0.1, 0.15, 0.25),  # 摆动阈值，ADR 倍数
    score_k=0.25,  # 打分、时段指标、特殊日检测用哪个阈值
    swing_skip_open_min=5,  # 打分只统计开盘这么多分钟以后开始的摆动（9:35 以后才交易）
    open_skip_compare=(0, 5, 15, 30, 90),  # 第 5 部分：分别剔除开盘这些分钟，各排一次名
    vwap_cross_min_hold=3,  # VWAP 穿越后至少保持的分钟数
    cost_tiers_bps=((20, 10), (100, 5), (None, 3)),  # 单边成本：< $20 10bp，< $100 5bp，其余 3bp
    halt_gap_minutes=10,  # 连续无成交超过该分钟数，标记疑似停牌（不剔除）
    exclude_early_close=True,  # 汇总时剔除提前收盘日
    min_dollar_vol=20e6,  # 成交额中位数低于此值标记 low_liquidity
    time_buckets_min=30,  # 时段分析的时间桶（分钟）
    print_filter_adr=0.3,  # 清洗：1 秒价格偏离附近中位数超过该 ADR 倍数视为离群成交
    anomaly_swing_density=7.5,  # 行为特殊日：Σ 摆动幅度 / 当日振幅超过此值
    anomaly_swing_multiple=4,  # 行为特殊日：摆动次数超过该证券中位数的倍数（且至少 10 次）
    exclude_data_issue_days=False,  # True = 数据异常日整天不参与汇总（默认只清洗、不剔除）
    # 杠杆 ETF: (标的, 名义倍数)；报表第 9 部分并排对照
    leveraged_pairs={"METU": ("META", 2), "TSLL": ("TSLA", 2), "NVDL": ("NVDA", 2)},
    include_extended_hours=True,  # 计算扩展时段参考指标（不参与打分）
    eth_min_bar_volume=100,  # 扩展时段成交量低于此值的 1 分钟 bar 不参与取价
    compare_periods=None,  # 多区间对比，例如 ("2026-04-01", "2026-06-30")
    # 合并区间：几个不相连的区间合成一个样本，例如
    # (("2026-01-02", "2026-03-31"), ("2026-07-01", "2026-09-30"))
    # 设了它，start_date / end_date 就不用了
    merge_periods=None,
    score_weights={"activity": 0.30, "rotation": 0.30, "swing_value": 0.25, "consistency": 0.15},
)
detail_tickers = None  # 单只证券详情；None = 综合得分前 3 名，也可以写 ["NVDA", "AMD"]

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from trader.analytics import build_report
from trader.analytics import report as R


def show(fig):
    display(fig)
    plt.close(fig)  # 已经显示过，避免单元格结束时再显示一次


rep = build_report(tickers, start_date, end_date, params)
for note in rep.notes:
    display(Markdown(f"> ⚠ {note}"))

## 1. 参数和数据摘要

In [ ]:
display(R.param_summary(rep))
if len(rep.excluded):
    print("未参与汇总的日子（按证券、原因计数）：")
    display(rep.excluded.groupby(["reason", "symbol"]).size().unstack(0).fillna(0).astype(int))
if len(rep.flagged_halts):
    print("疑似停牌（连续无成交超过阈值，只标记、不剔除）：")
    display(rep.flagged_halts)
if len(rep.data_issues):
    print(f"数据异常日 {len(rep.data_issues)} 个（已清洗，核对图在第 13 部分）：")
    display(rep.data_issues.round(2))
if len(rep.special_days):
    print(f"行为特殊日 {len(rep.special_days)} 个（数据没问题，只标记）：")
    display(rep.special_days.round(2))
print("各证券无成交分钟占比（中位数）：")
display(rep.medians[["missing_ratio"]].T.round(4))

## 2. 成本假设

In [ ]:
display(Markdown(R.COST_NOTE))

## 3. 排名总表
**按可交易性评分 `score_tradable` 排序**：可获取波动均值（绝对值，0.4）、转化效率（0.3）、每日转化效率中位数（0.3），
各项先在本批证券里做百分位排名。综合得分 `score_total` 和四个分项保留作参考（含摆动次数、VWAP 穿越等计数类指标）。
分数是本批证券里的相对排名（0–100）；灰底为参照证券，红字为成交额不足。
`_mean` 结尾的是均值，其余是中位数；`*_vs_spy` 是相对 SPY 的比值，可以跨批次比较。
摆动类指标只统计**开盘 5 分钟以后开始的**摆动（`swing_skip_open_min`），开盘段的另列在 `open_*` 两列，只作参考。
`conversion`（转化效率）= 可获取波动均值 / ADR%，旁边一列是可获取波动均值的绝对值（%）：转化效率高但绝对值很小的
（如 TLT、GLD），扣掉成本后空间很小。**可获取波动是比较用的相对指数**，没有计入反应延迟和滑点。

In [ ]:
R.styled_ranking(rep)

## 4. 摆动阈值敏感性
每个阈值下的摆动次数均值、可获取波动均值、可获取天数占比，以及用该阈值重算的可交易性评分和名次。
**名次随阈值变化不大，说明排名不依赖阈值的选择。**

In [ ]:
R.styled_sensitivity(rep)

## 5. 剔除开盘段的排名对照
分别只统计开始于开盘 0（不剔除）/ 5 / 15 / 30 / 90 分钟以后的摆动，各算一次可交易性评分和名次。
振幅、ADR、body_ratio 不变。**名次随剔除时长下降很多的证券，好成绩主要来自开盘段**；不做开盘的话要降级。
`capturable_mean` 是可获取波动均值的绝对值（%），看转化效率时一定同时看它。

In [ ]:
R.styled_open_skip(rep)

## 6. 象限散点图

In [ ]:
show(R.fig_quadrant(rep))

## 7. 分布对比（前 10 名和参照证券）

In [ ]:
show(R.fig_boxplots(rep))

## 8. 指标分布
第一版不输出"震荡型 / 趋势型"标签。看过这些分布后再定规则（倾向用相对 SPY 的比值）。

In [ ]:
show(R.fig_distributions(rep))

## 9. 时段热力图

In [ ]:
show(R.fig_heatmap(rep))

## 10. 杠杆 ETF 与标的对照

In [ ]:
if rep.leveraged.empty:
    print("没有设置 leveraged_pairs，或对应的证券不在本次计算里")
else:
    display(Markdown(R.LEVERAGE_NOTE))
    display(rep.leveraged.T.round(3))

## 11. 扩展时段参考（不参与打分）

In [ ]:
if rep.eth_summary.empty:
    print("没有计算扩展时段（include_extended_hours=False）")
else:
    display(Markdown(R.ETH_NOTE))
    display(rep.eth_summary.round(2))
    fig, pear, spear = R.fig_eth_scatter(rep)
    show(fig)
    print(f"盘前振幅与当天振幅的相关系数：Pearson {pear:.2f}，Spearman {spear:.2f}")

## 12. 单只证券详情
每日指标的时间序列；一个典型交易日（振幅和摆动次数最接近中位数的一天）的 1 分钟价格图，
红色实线连接识别出的摆动转折点，红色虚线是收盘时还没确认的最后一段。**请目测转折点是否合理。**
换一天看：`show(R.fig_detail_day(rep, "NVDA", date(2026, 9, 15)))`。

In [ ]:
for sym in detail_tickers or rep.detail_tickers():
    display(Markdown(f"### {sym}"))
    show(R.fig_detail_timeseries(rep, sym))
    show(R.fig_detail_day(rep, sym))
    ex = rep.extremes[rep.extremes["symbol"] == sym]
    print("振幅最大 / body_ratio 最高 / 未确认的最后一段最长的 3 天（多半是财报或重大消息日）：")
    display(ex.round(3))

## 13. 核对图
**数据异常日**：灰色竖线是清洗前的原始高低价，蓝色是清洗后的；看清洗掉的是不是明显的错误成交。
**行为特殊日**：数据没问题，只是那天走势特殊（安静来回震荡、财报、消息行情）。各最多 16 张，完整清单见第 1 部分。
发现新的坏数据时段，加到 `config/bad_data_windows.yaml`。

In [ ]:
for kind, title in (("data", "数据异常日"), ("special", "行为特殊日")):
    figs = R.figs_review_days(rep, kind)
    display(Markdown(f"### {title}"))
    if not figs:
        print("没有")
    for fig in figs:
        show(fig)

## 14. 多区间对比

In [ ]:
if rep.compare is None:
    print('没有设置 compare_periods（例如 compare_periods=("2026-04-01", "2026-06-30")）')
else:
    print(f"两个区间名次的 Spearman 相关系数：{rep.compare_spearman:.2f}（越接近 1 越稳定）")
    display(rep.compare.round(1))

## 15. 指标说明

In [ ]:
display(Markdown(R.METRIC_NOTES))

## 导出
CSV（Excel 可以直接打开）和摆动明细 Parquet 写到 `reports/intraday_vol_<开始>_<结束>/`。
要导出整份 HTML 报告：先保存 notebook（Ctrl+S），再把下面的 `export_html` 改成 `True` 运行这个单元格。

In [ ]:
out = rep.export()
print("已导出到", out)

export_html = False
if export_html:
    import subprocess
    import sys

    subprocess.run(
        [
            sys.executable, "-m", "nbconvert", "--to", "html", "--no-input", "日内波动报表.ipynb",
            "--output-dir", str(out), "--output", f"report_{rep.start}_{rep.end}",
        ],
        check=True,
    )  # fmt: skip
    print("HTML 报告：", out / f"report_{rep.start}_{rep.end}.html")